## Problema
Muitos clientes desejam revisar seus documentos de forma automatizada para garantir que estejam em conformidade com as normas da LGPD (Lei Geral de Proteção de Dados).
## Solução
Criação de um sistema multiagente capaz de:
1. Verificar documentos para garantir conformidade com as normas da LGPD.
2. Tratar documentos que não estejam em conformidade, anonimizando dados sensíveis.
3. Devolver os documentos revisados e tratados aos clientes.
## Agentes Utilizados
* Agente 1: Especialista em LGPD
  - **Função:** Verificar e analisar trechos do documento que não estão em conformidade com as normas da LGPD.
  - **Responsabilidade:** Identificar dados sensíveis e áreas que precisam de ajustes para atender às regulamentações.
* Agente 2: Anonimizador de Dados
  - **Função:** Anonimizar dados sensíveis presentes no documento, substituindo-os por caracteres genéricos (ex: "XXX").
  - **Responsabilidade:** Garantir que todos os dados identificados pelo Agente 1 sejam devidamente tratados.
* Agente 3: Revisor e Formalizador
  - **Função:** Verificar se o Agente 2 anonimizou corretamente os dados com base na análise do Agente 1. Além disso, aprimorar o texto, tornando-o mais formal e adequado ao ambiente jurídico.
  - **Responsabilidade:** Garantir que o documento final esteja em conformidade com a LGPD, com texto claro, formal e profissional.

## Initial Imports

In [9]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from utils import load_env
load_env()

import yaml
from crewai import Agent, Task, Crew

In [10]:
# type: ignore
import os
from utils import get_openai_api_key
from utils import get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## Loading Tasks and Agents YAML files

In [11]:
files = {
  'agents': 'configs/agents.yaml',
  'tasks': 'configs/tasks.yaml'
}

configs = {}
for config_type, file_path in files.items():
  with open(file_path, 'r') as config_file:
    configs[config_type] = yaml.safe_load(config_file)
    
agents_config = configs['agents']
tasks_config = configs['tasks']

## Importing Tools

In [12]:
from crewai_tools import (
  SerperDevTool, 
  ScrapeWebsiteTool,
  FileReadTool,
  MDXSearchTool
)

ferramenta_busca = SerperDevTool()
ferramenta_aprofundamento = ScrapeWebsiteTool()
ferramenta_leitura_arquivo = FileReadTool(file_path="./fake_document.md")
ferramenta_busca_semantica = MDXSearchTool(mdx="./fake_document.md")

In [13]:
from IPython.display import Markdown, display
display(Markdown("./fake_document.md"))

# CONTRATO DE PRESTAÇÃO DE SERVIÇOS

**Partes:**

- **Contratante:** João da Silva, CPF: 123.456.789-00, RG: 12.345.678-X, residente à Rua dos Exemplos, nº 123, Bairro Centro, Cidade Exemplo, Estado XP, CEP 12345-678.
- **Contratada:** Empresa Exemplo Ltda., CNPJ: 12.345.678/0001-90, com sede na Avenida das Empresas, nº 456, Bairro Comercial, Cidade Exemplo, Estado XP, CEP 98765-432.

**Objeto do Contrato:**

O presente contrato tem por objeto a prestação de serviços de consultoria jurídica pela Contratada à Contratante, conforme as condições estabelecidas neste instrumento.

**Cláusula 1 - Do Prazo:**

O contrato terá vigência de 12 (doze) meses, contados a partir da data de assinatura pelas partes.

**Cláusula 2 - Das Obrigações da Contratante:**

A Contratante compromete-se a fornecer todas as informações necessárias para a prestação dos serviços, incluindo:

- Informações pessoais, como nome completo, documentos de identificação (CPF, RG), endereço completo, telefone de contato.
- Dados financeiros, como extratos bancários, faturas de cartão de crédito, histórico de pagamentos.
- Dados médicos e de saúde, incluindo laudos e relatórios médicos, conforme necessário para análise jurídica.

**Cláusula 3 - Das Obrigações da Contratada:**

A Contratada compromete-se a manter o sigilo absoluto sobre todas as informações recebidas da Contratante, em conformidade com a legislação aplicável, incluindo, mas não se limitando à Lei Geral de Proteção de Dados (LGPD - Lei nº 13.709/2018).

**Cláusula 4 - Do Sigilo e Proteção de Dados:**

A Contratada declara que adota medidas técnicas e administrativas adequadas para proteger os dados pessoais sensíveis fornecidos pela Contratante, incluindo:

- Armazenamento seguro de documentos em formato físico e digital.
- Restrição de acesso aos dados apenas aos profissionais autorizados.
- Descarte seguro de documentos físicos e digitais ao final da prestação dos serviços.

**Cláusula 5 - Penalidades:**

Em caso de violação das obrigações de sigilo ou de proteção de dados, a Contratada estará sujeita às penalidades previstas na LGPD, bem como à indenização por danos morais e materiais que venham a ser causados à Contratante.

**Cláusula 6 - Da Rescisão:**

O contrato poderá ser rescindido por qualquer uma das partes, mediante aviso prévio de 30 (trinta) dias, sem prejuízo de eventuais indenizações cabíveis em caso de descumprimento de suas cláusulas.

**Assinaturas:**

Cidade Exemplo, 01 de fevereiro de 2025.

---

**João da Silva**  
Contratante

---

**Empresa Exemplo Ltda.**  
Contratada


## Configure Agents and Crew

In [14]:
# type: ignore
# Creating Agents
agente_especialista_classificador_lgpd = Agent(
  config=agents_config["agente_especialista_classificador_lgpd"],
  tools=[
    ferramenta_busca, 
    ferramenta_aprofundamento, 
    ferramenta_leitura_arquivo, 
    ferramenta_busca_semantica
    ]
)

agente_anonimizador = Agent(
  config=agents_config["agente_anonimizador"]
)

agente_revisador_lgpd = Agent(
  config=agents_config["agente_revisador_lgpd"],
  tools=[
    ferramenta_busca,
    ferramenta_aprofundamento
  ]
)

# Creating Tasks
task_especialista_classificador_lgpd = Task(
  config=tasks_config["task_especialista_classificador_lgpd"],
  agent=agente_especialista_classificador_lgpd
)

task_anonimizador = Task(
  config=tasks_config["task_anonimizador"],
  agent=agente_anonimizador,
  context=[task_especialista_classificador_lgpd]
)

task_revisador_lgpd = Task(  
  config=tasks_config["task_revisador_lgpd"],
  agent=agente_revisador_lgpd,
  context=[task_especialista_classificador_lgpd, task_anonimizador],
  output_file="documento_atualizado.md" 
)

# Creating Crew
crew = Crew(
  agents=[
    agente_especialista_classificador_lgpd,
    agente_anonimizador,
    agente_revisador_lgpd
  ],
  tasks=[
    task_especialista_classificador_lgpd,
    task_anonimizador,
    task_revisador_lgpd
  ]
)

2025-01-31 14:51:54,792 - 6928 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


LLM value is a string
LLM value is a string
LLM value is a string


## Kicking off the crew

In [15]:
# Run the crew
result = crew.kickoff()

# Agent: Especialista em ClassificaÃ§Ã£o de Documentos e Conformidade com a LGPD.
## Task: Realizar uma anÃ¡lise detalhada do documento, verificando sua conformidade com a Lei Geral de ProteÃ§Ã£o de Dados (LGPD).  Para isso, o agente deve pesquisar as normas e diretrizes da LGPD, identificar possÃ­veis violaÃ§Ãµes ou inconformidades no documento,  e justificar de forma clara e embasada os pontos que nÃ£o estÃ£o alinhados com a legislaÃ§Ã£o. O documento original deve ser preservado,  e as observaÃ§Ãµes devem ser adicionadas de forma estruturada ao final.



# Agent: Especialista em ClassificaÃ§Ã£o de Documentos e Conformidade com a LGPD.
## Thought: I need to start by reading the content of the document that requires analysis for compliance with the LGPD. I will proceed to read the file.
## Using tool: Read a file's content
## Tool Input: 
"{\"file_path\": \"./fake_document.md\"}"
## Tool Output: 
# CONTRATO DE PRESTAÃ‡ÃƒO DE SERVIÃ‡OS

**Partes:**

- **Contratante:** JoÃ£o da Silva, CP

In [16]:
from IPython.display import Markdown, display
display(Markdown("./documento_atualizado.md"))

# CONTRATO DE PRESTAÇÃO DE SERVIÇOS

**Partes:**

- **Contratante:** XXX, CPF: XXX, RG: XXX, residente à XXX, nº XXX, Bairro XXX, Cidade XXX, Estado XXX, CEP XXX.
- **Contratada:** XXX, CNPJ: XXX, com sede na XXX, nº XXX, Bairro XXX, Cidade XXX, Estado XXX, CEP XXX.

**Objeto do Contrato:**

O presente contrato tem por objeto a prestação de serviços de consultoria jurídica pela Contratada à Contratante, conforme as condições estabelecidas neste instrumento.

**Cláusula 1 - Do Prazo:**

O contrato terá vigência de 12 (doze) meses, contados a partir da data de assinatura pelas partes.

**Cláusula 2 - Das Obrigações da Contratante:**

A Contratante compromete-se a fornecer todas as informações necessárias para a prestação dos serviços, incluindo:

- Informações pessoais, tais como nome completo, documentos de identificação (CPF, RG), endereço completo e telefone de contato.
- Dados financeiros, como extratos bancários, faturas de cartão de crédito e histórico de pagamentos.
- Dados médicos e de saúde, incluindo laudos e relatórios médicos, conforme necessário para análise jurídica, desde que com o consentimento explícito do titular.

**Cláusula 3 - Das Obrigações da Contratada:**

A Contratada compromete-se a manter o sigilo absoluto sobre todas as informações recebidas da Contratante, em conformidade com a legislação aplicável, incluindo, mas não se limitando à Lei Geral de Proteção de Dados (LGPD - Lei nº 13.709/2018).

**Cláusula 4 - Do Sigilo e Proteção de Dados:**

A Contratada declara que adota medidas técnicas e administrativas adequadas para proteger os dados pessoais sensíveis fornecidos pela Contratante, incluindo:

- Armazenamento seguro de documentos em formato físico e digital.
- Restrição de acesso aos dados apenas aos profissionais autorizados.
- Descarte seguro de documentos físicos e digitais ao final da prestação dos serviços.
- Elaborar e implementar políticas de segurança da informação a fim de garantir a integridade e a confidencialidade dos dados tratados.

**Cláusula 5 - Penalidades:**

Em caso de violação das obrigações de sigilo ou de proteção de dados, a Contratada estará sujeita às penalidades previstas na LGPD, bem como à indenização por danos morais e materiais que venham a ser causados à Contratante.

**Cláusula 6 - Da Rescisão:**

O contrato poderá ser rescindido por qualquer uma das partes, mediante aviso prévio de 30 (trinta) dias, sem prejuízo de eventuais indenizações cabíveis em caso de descumprimento de suas cláusulas.

**Assinaturas:**

Cidade XXX, 01 de fevereiro de 2025.

---

**XXX**  
Contratante

---

**XXX**  
Contratada
```